# 12 - Prepare Human Evaluation

### Setup

In [1]:
import os
import pandas as pd
import numpy as np
from huggingface_hub import HfApi

## 1. Konfigurasi

In [2]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_Faiss")
    except ImportError:
        token = os.environ.get("HF_Faiss")
        if token is None:
            raise RuntimeError("HF_TOKEN tidak ditemukan.")
        return token
 
HF_TOKEN = get_hf_token()
HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else "/workspace")
 
MODEL_KEYS = ["gemma2", "llama3.2"]
SAMPLE_FRACTION = 0.01  
SEED = 42
N_EVALUATORS = 5 

## 2. Siapkan data

In [3]:
# ==== AMBIL SUBSET DARI HASIL PREDIKSI TIAP MODEL ====
 
import uuid
 
all_rows = []
for model_key in MODEL_KEYS:
    predictions_path = f"hf://datasets/{HF_REPO}/eval_{model_key}_predictions.csv"
    df = pd.read_csv(predictions_path, storage_options={"token": HF_TOKEN})
 
    n_sample = max(1, round(len(df) * SAMPLE_FRACTION))
    print(f"{model_key}: total {len(df)} | 5% sample: {n_sample}")
 
    sample_df = df.sample(n=min(n_sample, len(df)), random_state=SEED).reset_index(drop=True)
 
    for _, row in sample_df.iterrows():
        for condition, pred_col in [("base", "ans_pred_base"), ("finetuned", "ans_pred_finetuned")]:
            all_rows.append({
                "eval_id": uuid.uuid4().hex[:12],  # anonim, TIDAK mengandung info model/kondisi
                "_model_key": model_key,       # hidden ground truth, kolom berawalan _ jangan dikirim ke evaluator
                "_condition": condition,        # hidden ground truth
                "instruction": row["instruction"],
                "ans_ref": row["ans_ref"],
                "ans_pred": row[pred_col],
                "fluency": "",
                "factual_correctness": "",
                "completeness": "",
            })
 
master_df = pd.DataFrame(all_rows)

gemma2: total 1791 | 5% sample: 18
llama3.2: total 1791 | 5% sample: 18


In [4]:
# ==== ACAK URUTAN (blind terhadap model & kondisi) ====
 
master_df = master_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
master_df.insert(0, "item_no", range(1, len(master_df) + 1))
 
print(f"Total item dinilai per evaluator: {len(master_df)} ({len(MODEL_KEYS)} model x 2 kondisi x sample per model)")

Total item dinilai per evaluator: 72 (2 model x 2 kondisi x sample per model)


In [5]:
# ==== VERSI EVALUATOR (kolom hidden dibuang) ====
 
BLIND_COLUMNS = ["item_no", "eval_id", "instruction", "ans_ref", "ans_pred",
                 "fluency", "factual_correctness", "completeness"]
blind_df = master_df[BLIND_COLUMNS]

In [6]:
# ==== VERSI KUNCI (untuk analisis) ====
 
key_df = master_df[["item_no", "eval_id", "_model_key", "_condition"]]

## 3. Buat Salinan

In [7]:
# ==== SIMPAN & PUSH ====
 
api = HfApi(token=HF_TOKEN)
 
for evaluator_id in [f"evaluator{i+1}" for i in range(N_EVALUATORS)]:
    output_path = f"{OUTPUT_DIR}/human_eval_{evaluator_id}.csv"
    blind_df.to_csv(output_path, index=False)
    api.upload_file(
        path_or_fileobj=output_path,
        path_in_repo=f"human_eval_{evaluator_id}.csv",
        repo_id=HF_REPO,
        repo_type="dataset",
    )
    print(f"Ter-push (BLIND, untuk evaluator): {HF_REPO}/human_eval_{evaluator_id}.csv")
 
key_path = f"{OUTPUT_DIR}/human_eval_key.csv"
key_df.to_csv(key_path, index=False)
api.upload_file(
    path_or_fileobj=key_path,
    path_in_repo="human_eval_key.csv",
    repo_id=HF_REPO,
    repo_type="dataset",
)
print(f"Ter-push (KUNCI, JANGAN dibagikan ke evaluator): {HF_REPO}/human_eval_key.csv")
 
print("Skala Likert 1-5 untuk fluency, factual_correctness, completeness sesuai Tabel II proposal.")

Ter-push (BLIND, untuk evaluator): Makaareeem/publikasi-rag-finetuning-dataset/human_eval_evaluator1.csv
Ter-push (BLIND, untuk evaluator): Makaareeem/publikasi-rag-finetuning-dataset/human_eval_evaluator2.csv
Ter-push (BLIND, untuk evaluator): Makaareeem/publikasi-rag-finetuning-dataset/human_eval_evaluator3.csv
Ter-push (BLIND, untuk evaluator): Makaareeem/publikasi-rag-finetuning-dataset/human_eval_evaluator4.csv
Ter-push (BLIND, untuk evaluator): Makaareeem/publikasi-rag-finetuning-dataset/human_eval_evaluator5.csv
Ter-push (KUNCI, JANGAN dibagikan ke evaluator): Makaareeem/publikasi-rag-finetuning-dataset/human_eval_key.csv
Skala Likert 1-5 untuk fluency, factual_correctness, completeness sesuai Tabel II proposal.
